# The correction operator scored against tide gauges

Before estimating anything from imagery we asked a cheaper question:
is an estuarine transfer operator worth having at all? Pairs of
existing tide gauges answer it for free — fit the transfer on one
stretch of days, score it on held-out days.

The first version of that validation over-credited the operator badly.
At Ferrol it reported a quarter-metre of improvement, and nearly all of
it turned out to be harmonic filtering of storm surge — something any
harmonic reconstruction does, transfer or not. The honest scoreboard
therefore carries a middle term (harmonics only, no transfer), and only
the difference between that column and the operator column belongs to the operator. Formally, with a harmonic
reconstruction $H(t)$ of the outer gauge and the transfer applied per
constituent (gain $(1+\gamma_k)$, lag $\Delta t_k$), the three scores are
$\mathrm{RMSE}(\text{model})$, $\mathrm{RMSE}(H)$ and
$\mathrm{RMSE}(T[H])$, all median-centred (datum-free) on the same
held-out window; the operator's real contribution is
$\mathrm{RMSE}(H)-\mathrm{RMSE}(T[H])$, nothing else.

**You are here: 03.** The measured clocks become a correction operator; here it is scored against real tide gauges it never saw in calibration.

```text
+- the evidence chain ------------------------------------------------+
|  datacube -> water masks -> per-pixel wet/dry series                |
|    01 what is estimable  ->  02 boundary audit  ->  03 operator vs  |
|    gauges  ->  04 elevations vs truth  ->  05 uncertainty and       |
|    hydraulic layers  ->  06 negatives kept  ->  07 coast census     |
+---------------------------------------------------------------------+
```

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# run from the repo root so the results/ paths resolve
here = Path.cwd()
while not (here / "pyintertidal").is_dir():
    if here.parent == here:
        raise FileNotFoundError("repo root not found above " + str(Path.cwd()))
    here = here.parent
os.chdir(here)

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

### What "the operator" is

The correction operator $T$ is the recipe that turns the mouth's tide
series into the interior's: per tidal constituent $k$, an amplitude gain
$(1+\gamma_k)$ and a time lag $\Delta t_k$ — because friction and
convergence act on each frequency differently (that is why the interior
wave is asymmetric: the overtides grow faster than M2). It can be fitted
from a pair of gauges (`from_gauges`, used in this notebook as the
yardstick) or carried by MAREA's imagery-measured phase with gain pinned
to the boundary model (the zero-instrument rows). The scoreboard below
never lets a column take credit for work it did not do.

## 1. Gauge pairs around the world: controls, estuaries, one broken pair
Controls (two harbour stations with no estuary between them) should
show no operator gain, and don't. Short deep rias show millimetres.
Shallow constricted mouths show real gains at only 2 km. One pair had
to be discarded for data reasons (metres of disagreement between
stations 6 km apart — a datum or sensor problem, not physics), and one
apparent control turned out to be an honest propagation case. What
decides the gain is the character of the estuary, not the distance:

In [2]:
# the honest scoreboard: "harmonics" already includes everything a plain
# harmonic fit gives (surge filtering included); only the difference to
# the operator column is attributable to the estuarine transfer
t = load("results/p7_tabla_mareografos/result.json")["tabla"]
print(f"{'pair':46s} {'harmonics':>10s} {'operator':>10s} {'op. gain':>9s}")
for k, v in t.items():
    m = v.get("rmse_m", {})
    a = m.get("historico_solo_armonicos")
    b = m.get("historico_con_operador_mareografos")
    if a is None or b is None:
        continue
    print(f"{k:46s} {a:10.4f} {b:10.4f} {a - b:+9.4f}")

pair                                            harmonics   operator  op. gain
Honolulu - Keehi (control, 2.7 km)                 0.0307     0.0307   -0.0000
Barcelona - Port (control, 2.9 km)                 0.1176     0.1175   +0.0001
Ferrol interior (ria profunda, 6.4 km)             0.1159     0.1071   +0.0089
Terneuzen (Escalda, 28 km)                         0.4051     0.3072   +0.0979
Sevilla (Guadalquivir, 85 km)                      1.0629     0.2608   +0.8021
Dieppe (costa abierta, 90 km)                      1.0719     0.3850   +0.6868


## 2. Contraction gate on the synthetic world
Does the operator recover damage it is meant to fix, and is it a fixed
point of its own estimation?

In [3]:
r = load("results/m4_gate_sim/result.json")
print("elevation RMSE per band, uniform -> operator:")
for c, u, o in zip(r["centros_km"], r["rmse_z"]["uniforme"],
                   r["rmse_z"]["operador"]):
    print(f"  s={c:5.2f} km: {u:.3f} -> {o:.3f}")
print("damage recovered:", round(r["fraccion_dano_recuperado"], 2),
      "| gate:", r["puerta"])

elevation RMSE per band, uniform -> operator:
  s= 1.80 km: 0.544 -> 0.544
  s= 3.41 km: 0.245 -> 0.199
  s= 4.09 km: 0.267 -> 0.208
  s= 4.52 km: 0.305 -> 0.240
  s= 5.41 km: 0.352 -> 0.283
  s= 7.71 km: 0.440 -> 0.367
damage recovered: 1.95 | gate: {'muerde_ok': True, 'boca_ok': True, 'contraccion_ok': True, 'PASA': True}


## 3. Terneuzen: every candidate against a gauge held out of calibration

In [4]:
# sorted worst to best; every row scored on the same held-out window
r = load("results/p6_comparativa/result.json")
for k, v in sorted(r["rmse_m"].items(), key=lambda kv: -kv[1]):
    print(f"  {v:.4f} m  {k}")
print("gauge-calibrated reference (needs 2 gauges):",
      r["referencia_con_mareografos"])

  0.7051 m  GOT4.8_nc tal cual
  0.7029 m  GOT4.10_nc tal cual
  0.5916 m  ensemble (media de los 3)
  0.4195 m  EOT20 tal cual
  0.4096 m  EOT20 + adaptador v3 (tau=5m)
  0.4074 m  EOT20 + OPERADOR NUEVO M2a (tau=7.3m)
  0.4063 m  techo: mejor retardo contra el mareografo (+10.0m)
gauge-calibrated reference (needs 2 gauges): {'solo_armonicos': 0.4051, 'con_operador': 0.3072}


## 4. The Scheldt axis blind, and two sites kept as exploratory
The along-channel lag gradient measured from imagery alone is compared
with the gauge-measured one. The same machinery was also run at
Sheerness (Thames mouth) and Saint-Malo; without a gauge pair there is
no external truth at either, so both stay labelled exploratory rather
than being promoted:

In [5]:
r = load("results/p4_escalda/result.json")
print(f"Scheldt blind gradient: {r['gradiente_m2a_min_km']:.2f} min/km "
      f"vs gauges {r['gradiente_mareografos_min_km']} -> {r['validacion']}")
for s in ("sheerness", "stmalo"):
    e = load(f"results/p4_{s}/result.json")
    print(f"\n{s}: tau profile {np.round(e['tau_m2a_min'], 0)} min")
    print("  " + e["nota"])

Scheldt blind gradient: 0.72 min/km vs gauges 0.9 -> {'gradiente_ok': True, 'signo_ok': True, 'PASA': True}

sheerness: tau profile [ 0.  6.  8.  7.  1. -5.] min
  EXPLORATORIO: sin par de mareografos no hay verdad externa del gradiente; ancla en la abertura al mar

stmalo: tau profile [0. 8. 6. 0. 2. 2.] min
  EXPLORATORIO: sin par de mareografos no hay verdad externa del gradiente; ancla en la abertura al mar
